In [12]:
from sklearn.metrics import mean_squared_error

def get_rmse(R, P, Q, non_zeros):
    error = 0
    # 예측 행렬 R_hat = P * Q.T
    full_pred_matrix = np.dot(P, Q.T)

    # 실제 값 R에서 0이 아닌 값의 위치만 추출하여 오차 계산
    x_non_zero_ind = [non_zero[0] for non_zero in non_zeros]
    y_non_zero_ind = [non_zero[1] for non_zero in non_zeros]
    R_non_zeros = R[x_non_zero_ind, y_non_zero_ind]

    predicted_non_zeros = full_pred_matrix[x_non_zero_ind, y_non_zero_ind]

    mse = mean_squared_error(R_non_zeros, predicted_non_zeros)
    rmse = np.sqrt(mse)

    return rmse
def matrix_factorization(R, K, steps=200, learning_rate=0.01, r_lambda = 0.01 ):
  num_users, num_items = R.shape
  np.random.seed(1)
  P = np.random.normal(scale=1./K, size=(num_users, K))
  Q = np.random.normal(scale=1./K, size=(num_items, K))
  non_zeros = [ (i, j, R[i, j]) for i in range(num_users) for j in range(num_items) if R[i, j]>0]
  for step in range(steps):
    for i, j, r in non_zeros:
      eij = r - np.dot(P[i, :], Q[j, :].T)
      P[i, :] = P[i, :] + learning_rate*(eij * Q[j, :] - r_lambda*P[i, :])
      Q[j, :] = Q[j, :] + learning_rate*(eij * P[i, :] - r_lambda*Q[j, :])
    rmse = get_rmse(R, P, Q, non_zeros)
    if (step % 10) == 0 :
      print("### iteration step : ", step, " rmse : ", rmse)
  return P, Q


In [9]:
import os

# 현재 폴더에 무엇이 있는지 확인
print("현재 폴더 파일:", os.listdir('.'))

# 만약 폴더 안에 폴더가 있다면 전체를 뒤져서 movies.csv를 찾습니다.
for root, dirs, files in os.walk('/kaggle/working/'): # 캐글 기준
    for file in files:
        if file.endswith(".csv"):
            print(f"찾았습니다! 전체 경로: {os.path.join(root, file)}")

현재 폴더 파일: ['.config', 'movies.csv', 'tags.csv', 'links.csv', 'README.txt', 'ratings.csv', 'sample_data']


In [13]:
import pandas as pd
import numpy as np

movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')
ratings = ratings[['userId', 'movieId', 'rating']]
ratings_matrix = ratings.pivot_table('rating', index='userId', columns='movieId')

rating_movies = pd.merge(ratings, movies, on='movieId')
ratings_matrix = rating_movies.pivot_table('rating', index='userId', columns='title')

In [14]:
P, Q = matrix_factorization(ratings_matrix.values, K=50, steps=200, learning_rate=0.01, r_lambda = 0.01)
pred_matrix = np.dot(P, Q.T)

### iteration step :  0  rmse :  2.9023619751336867
### iteration step :  10  rmse :  0.7335768591017927
### iteration step :  20  rmse :  0.5115539026853442
### iteration step :  30  rmse :  0.37261628282537446
### iteration step :  40  rmse :  0.2960818299181014
### iteration step :  50  rmse :  0.2520353192341642
### iteration step :  60  rmse :  0.22487503275269854
### iteration step :  70  rmse :  0.2068545530233154
### iteration step :  80  rmse :  0.19413418783028685
### iteration step :  90  rmse :  0.18470082002720406
### iteration step :  100  rmse :  0.17742927527209104
### iteration step :  110  rmse :  0.1716522696470749
### iteration step :  120  rmse :  0.16695181946871726
### iteration step :  130  rmse :  0.16305292191997542
### iteration step :  140  rmse :  0.15976691929679646
### iteration step :  150  rmse :  0.1569598699945732
### iteration step :  160  rmse :  0.15453398186715425
### iteration step :  170  rmse :  0.15241618551077643
### iteration step :  180  rm

In [15]:
ratings_pred_matrix = pd.DataFrame(data=pred_matrix, index=ratings_matrix.index, columns=ratings_matrix.columns)
ratings_pred_matrix.head(3)

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,3.055084,4.092018,3.564130,4.502167,3.981215,1.271694,3.603274,2.333266,5.091749,3.972454,...,1.402608,4.208382,3.705957,2.720514,2.787331,3.475076,3.253458,2.161087,4.010495,0.859474
2,3.170119,3.657992,3.308707,4.166521,4.311890,1.275469,4.237972,1.900366,3.392859,3.647421,...,0.973811,3.528264,3.361532,2.672535,2.404456,4.232789,2.911602,1.634576,4.135735,0.725684
3,2.307073,1.658853,1.443538,2.208859,2.229486,0.780760,1.997043,0.924908,2.970700,2.551446,...,0.520354,1.709494,2.281596,1.782833,1.635173,1.323276,2.887580,1.042618,2.293890,0.396941


In [17]:
def get_unseen_movies(ratings_matrix, userId):
    # userId로 입력받은 사용자의 모든 영화 정보를 추출하여 Series로 반환함.
    # user_rating은 영화명을 인덱스로 가지는 Series 객체임.
    user_rating = ratings_matrix.loc[userId, :]

    # user_rating이 0보다 크면 기존에 관람한 영화임. 대상 index를 추출하여 list 객체로 만듦
    already_seen = user_rating[user_rating > 0].index.tolist()

    # 모든 영화명을 list 객체로 만듦.
    movies_list = ratings_matrix.columns.tolist()

    # list comprehension으로 already_seen에 해당하는 영화는 movies_list에서 제외함.
    unseen_list = [ movie for movie in movies_list if movie not in already_seen]

    return unseen_list

In [19]:
import pandas as pd

# 1. 추천 함수 정의 (사용자 ID를 기반으로 보지 않은 영화 중 예측 점수가 높은 영화 추출)
def recomm_movie_by_userid(pred_df, userId, unseen_list, top_n=10):
    # 예측 점수 행렬(pred_df)에서 해당 사용자의 데이터만 뽑아, 안 본 영화들의 점수를 내림차순 정렬
    recomm_movies = pred_df.loc[userId, unseen_list].sort_values(ascending=False)[:top_n]
    return recomm_movies

# 2. (위에서 정의한) get_unseen_movies 함수가 실행되어 있어야 합니다.
# 9번 사용자가 관람하지 않은 영화 리스트를 가져옵니다.
try:
    unseen_list = get_unseen_movies(ratings_matrix, 9)

    # 3. 함수 호출 (이제 여기서 NameError가 나지 않습니다)
    recomm_movies_series = recomm_movie_by_userid(ratings_pred_matrix, 9, unseen_list, top_n=10)

    # 4. 결과를 보기 좋게 데이터프레임으로 변환
    recomm_movies = pd.DataFrame(data=recomm_movies_series.values,
                                 index=recomm_movies_series.index,
                                 columns=['pred_score'])

    print("9번 사용자를 위한 추천 영화 TOP 10:")
    print(recomm_movies)

except NameError as e:
    print(f"에러 발생: {e}. 'get_unseen_movies' 함수가 들어있는 셀을 먼저 실행했는지 확인해주세요.")

9번 사용자를 위한 추천 영화 TOP 10:
                                                    pred_score
title                                                         
Rear Window (1954)                                    5.704612
South Park: Bigger, Longer and Uncut (1999)           5.451100
Rounders (1998)                                       5.298393
Blade Runner (1982)                                   5.244951
Roger & Me (1989)                                     5.191962
Gattaca (1997)                                        5.183179
Ben-Hur (1959)                                        5.130463
Rosencrantz and Guildenstern Are Dead (1990)          5.087375
Big Lebowski, The (1998)                              5.038690
Star Wars: Episode V - The Empire Strikes Back ...    4.989601


In [20]:
unseen_list = get_unseen_movies(ratings_matrix, 9)
recomm_movies = recomm_movie_by_userid(ratings_pred_matrix, 9, unseen_list, top_n=10)
recomm_movies = pd.DataFrame(data=recomm_movies.values, index=recomm_movies.index, columns=['pred_score'])
recomm_movies

,pred_score
title,
Rear Window (1954),5.704612
"South Park: Bigger, Longer and Uncut (1999)",5.451100
Rounders (1998),5.298393
Blade Runner (1982),5.244951
Roger & Me (1989),5.191962
Gattaca (1997),5.183179
Ben-Hur (1959),5.130463
Rosencrantz and Guildenstern Are Dead (1990),5.087375
"Big Lebowski, The (1998)",5.038690
